# Phase6: 評価・検証レポート — プロジェクトの総括

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: ここまでの総括。どの条件なら使えて、どこから使えないか（運用できる範囲）をまとめる回です。
> - **たとえ話**: 通知表。何が得意で、何が苦手かを一覧にする。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「5（結果を読むときの心得）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> ⏱ 所要時間: 説明を読む 40分 ＋ 実行 約30秒

## このプロジェクトのゴール（おさらい）
駐車場の**ナンバー読み取り精算システム**のミニ版を、合成画像だけで作り、**「AIはシステムの一部品」**であることを、自分の手で確かめる。

| Phase | やったこと |
|---|---|
| 0 | 練習用の**合成画像と正解**を作る（難易度3段階） |
| 1 | **AIなし**（古典的な画像処理）のベースライン |
| 2 | 板の**四隅を学ぶAI**（検出） |
| 3 | 文字を**読むAI**（CRNN＋CTC） |
| 4 | 書式検査・確信度・「自動処理／要確認／読み取り不可」の**振り分け** |
| 5 | 入庫の記録・出庫の照合・料金計算（**システム全体**）。誤課金・人手の測定 |

## このNotebookのゴール
プロジェクト全体を**1つのレポート**にまとめます。

1. **最終評価**（これまで一度も使っていない **test データ**）の結果を読む
2. 開発中の見積もり（val）と、最終評価（test）が**一致するか**を確かめる
3. **混同行列**（どの文字を取り違えるか）、**失敗例**を見る
4. **劣化要因の単独アブレーション**で、システムの**運用できる範囲**（カメラの仕様の目安）を知る
5. **限界**と**学び**を整理する

## なぜ「最終評価は1回だけ」なのか
評価データで何度も試し、**結果を見て調整**すると、そのデータに**合わせてしまい**、評価が甘くなります（試験問題を見ながら勉強するのと同じ）。
そこで、データを3つに分けていました。

| データ | 使い方 |
|---|---|
| train | 学習（その場で生成） |
| val（と較正セット） | 開発中に、何度でも使って調整（しきい値・エポック・手法の選択） |
| **test** | **最後に1回だけ**。結果を見て調整しない |

今回、**test は最終評価でちょうど1回だけ**使いました。スクリプトにも「**使用済みの記録**」を持たせ、2回目は実行できないようにしてあります。

## キーワード
| 用語 | 意味 |
|---|---|
| **最終評価（test）** | 一度も調整に使っていないデータでの、最後の評価 |
| **混同行列** | 「正解の文字」と「読んだ文字」の対応表。どの文字を取り違えやすいかが分かる |
| **アブレーション** | 1つずつ要因を変えて、効果を調べる実験 |
| **運用できる範囲** | システムが十分な精度で動く条件の範囲（カメラの仕様の目安） |
| **sim2realギャップ** | 合成と、現実の違い。合成での精度が、現実で保証されない理由 |

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import PIL, numpy, pandas  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"必要なライブラリが見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (plate_reader_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
%matplotlib inline
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
FIN, REP, DEV = ROOT / "outputs" / "final", ROOT / "outputs" / "report", ROOT / "outputs" / "final_dev"
IMG = ROOT / "report_images"          # 主な図（リポジトリに含まれる）
def show(name):
    f = (REP / name) if (REP / name).exists() else (IMG / name)
    display(IPImage(filename=str(f))) if f.exists() else print(f"{name} がありません（src/eval_report.py を実行してください）")
if not (FIN / "table_path.csv").exists():
    print("最終評価の結果がありません。このリポジトリの outputs/final/ は git に含まれません。再現するには Phase2〜4 の学習の後、src/final_eval.py（予行演習=val）を実行してください。")

上のセルは、道具の読み込みです。エラーが出たら、右上のカーネルを **Python (plate_reader_lab)** に切り替えてください。

## 1. test の使用記録 — 本当に1回だけか

最終評価を実行したときに書かれた**使用済みの記録**を見ます。**モデルとしきい値は、評価の前に決まっていた**（モデルの指紋＝ハッシュも記録）ことの証拠です。

In [ ]:
f = FIN / "TEST_USED.json"
if f.exists():
    info = json.loads(f.read_text(encoding="utf-8"))
    for k, v in info.items():
        print(f"{k}: {v}")

**見るポイント**
- **実行日時**と、**gitコミット**（そのときのコードの状態）が残っています。
- **しきい値**と**モデルのハッシュ**は、評価の**前**に決めたものです（しきい値は「較正セット」で決めた。testは使っていない）。
- このファイルがある間、`final_eval.py --split test` は**実行を拒否**します。

## 2. 最終評価（test）の結果 — システムの道のり

4つの構成を、**同じtest画像**で比べます（完全一致の割合。**各難易度1,000枚**）。

| 構成 | 内容 |
|---|---|
| **P1** | 古典的な検出＋古典的なOCR（機械学習なし。Phase1） |
| **P2** | AIの検出＋古典的なOCR（Phase2） |
| **P3** | AIの検出＋CRNN（Phase3）。CRNNは3つのシードの平均 |
| （参考） | CRNN・**正解の四隅**を使った場合（検出の誤りを除いた、読み取りの力そのもの） |

| 難易度 | 枚数 | P1 古典（検出＋OCR） | P2 AI検出＋古典OCR | P3 AI検出＋CRNN | (参考)CRNN・正解の四隅 | P3の標準偏差 |
|---|---|---|---|---|---|---|
| easy | 1000 | 0.710 | 0.937 | 0.980 | 0.981 | 0.015 |
| medium | 1000 | 0.154 | 0.356 | 0.895 | 0.940 | 0.009 |
| hard | 1000 | 0.001 | 0.000 | 0.085 | 0.209 | 0.006 |

In [ ]:
show("path_test.png")

### 何が分かるか
1. **各段階で、はっきり良くなっています。**
   - **Medium**: 0.15（古典）→ 0.36（AI検出＋古典OCR）→ **0.90（AI検出＋CRNN）**。**検出のAI化**と、**読み取りのAI化**が、**それぞれ**大きく効いた。
   - **Easy**: 0.71 → 0.94 → **0.98**。きれいな条件では、元々ほぼ読めたが、**検出の位置精度**で差がついた。
   - **Hard**: **0.09**。AIでも、ほとんど読めない（**画像から情報が失われている**。Phase3の結論）。
2. **「正解の四隅」を使うと、Hard は 0.21、Medium は 0.94。** 検出の誤差が、まだ少し残っている（特にHard）。
3. **開発時（val）の見積もりと、最終（test）がほぼ一致**（右の図）。差は**±0.01以内**（P3）。**開発中の評価が、未使用データでも信頼できた**ということです。

> 💡 もし test の結果が val より大きく悪かったら、**開発中に val に合わせすぎていた**（過学習の一種）と分かります。今回はそうなっていませんでした。

## 3. 検出と読み取りの関係

最終評価（test）での、検出の結果です。

| 難易度 | 検出率(IoU≥0.5) | Det@0.9 | 平均IoU | 四隅誤差(px) | 古典の検出率(IoU≥0.5) |
|---|---|---|---|---|---|
| easy | 1.000 | 0.998 | 0.966 | 1.557 | 0.905 |
| medium | 1.000 | 0.824 | 0.928 | 2.860 | 0.760 |
| hard | 0.998 | 0.196 | 0.849 | 5.058 | 0.402 |

- **検出率（IoU≥0.5）は、全難易度でほぼ100%**（古典は Hard で 0.40）。
- しかし**読み取りに必要な位置精度**（**Det@0.9**）は、Hard で **0.20**。**四隅誤差 約5px** で、読み取りの限界（誤差が1〜2px以内なら読める。Phase2）を超えている。
- **Phase2・Phase3 の結論どおり**です（test でも再現した）。

## 4. 混同行列 — どの文字を取り違えるか

**正解の文字（縦）→ 読んだ文字（横）**の対応表です（色が濃いほど、回数が多い）。**較正セット**（3,000枚×3シード）の読み取り結果で作りました（枚数が多く、安定するため）。左が Medium、右が Hard。

**数字・記号**

In [ ]:
show("confusion_digits.png")

**見るポイント**
- **Medium（左）**: 取り違えは**少なく**、**一番多いのは「脱落」**（右端の列。6、9、8が読み落とされる）。取り違えは、**6↔5、6↔8、5↔6**など、**形が似た数字**に集中。
- **Hard（右）**: **ほぼ全部の数字が、ほぼ全部の数字と取り違え**られている。特に**7↔1、6↔5、8↔6、9↔5、8↔9**。**ぼけ・ノイズ・低解像度で、数字の細部が失われる**ためで、**人間が見ても区別しにくい**。
- 「-」（ハイフン）と「・」は、ほとんど取り違えない（形が全然違うから）。

**かな**

In [ ]:
show("confusion_kana.png")

**見るポイント**: かなも、**形が似たもの**（な↔は、ま↔あ、た↔か、か↔た）の取り違えが中心です。Hard では、かなも広く取り違えます。

### AIの限界を表す、1つの視点
**取り違えの傾向は、人間の感覚と似ています**（形が似ているものを間違える）。ですから、「**どれが間違えやすいか**」を知っておけば、**要確認の画面に、その候補を出す**（例: 「6か5か8か」）といった、**人の確認を助ける工夫**ができます。

## 5. 劣化要因の単独アブレーション — 運用できる範囲

**Easyの基準**（板の大きさ40%、劣化なし）に、**劣化を1つだけ**足して、**AI**と**古典**がどう崩れるかを調べました（各点80枚）。**「どの条件で、どこまで読めるか」**が分かります。

In [ ]:
show("factor_ablation.png")

**見るポイント**
- **AI（青）は、多くの劣化に強い**: ノイズ（σ=45でもほぼ満点）、逆光・影、汚れ、雨滴、JPEG圧縮。**古典（オレンジ）は、ぼけ・逆光・汚れ・透視で、すぐ崩れる**。
- **AIが急に崩れる条件**: **板が小さい**（画面幅の20%）、**回転が大きい**（15°以上）、**透視の歪みが大きい**（0.1以上）、**線による隠れ**、**暗すぎ**（明るさ0.2）、**ぼけ・動きぶれが大きい**。
- **これらは、学習データの範囲の端（または外）**です（たとえば Hard の板の大きさは画面幅の20〜35%、回転は±12°まで）。**学習した範囲の外では、急に崩れる**（Phase2・3の分布シフトと同じ）。
- **古典のほうが強い条件**もあります: **回転**（25°でも約0.5）、**小さな板**（20%）。AIは、そこに**弱点がある**。**「AIのほうが常に良い」わけではない**。

### 運用できる範囲の目安（AIの完全一致が0.9以上を保つ強さ）

| 要因 | AI（0.9を下回る最初の強さ） | 古典 | AIが0.9以上を保つ最後の強さ |
|---|---|---|---|
| ノイズ σ | — | 0 | 45 |
| ぼけ σ | 3 | 0 | 2 |
| 動きぶれ(px) | 6 | 0 | 4 |
| 明るさ | 0.2 | 1 | 0.3 |
| コントラスト | 0.3 | 1 | 0.4 |
| 逆光・影の強さ | — | 0 | 0.9 |
| 汚れの数 | — | 0 | 6 |
| 雨滴の数 | — | 0 | 80 |
| 隠れ(線の数) | 2 | 0 | 1 |
| 低解像度(縮小率) | 4 | 1 | 3 |
| JPEG品質 | — | 95 | 8 |
| 回転(度) | 15 | 0 | 10 |
| 透視の歪み | 0.1 | 0 | 0.05 |
| 板の大きさ(画面幅比) | 0.2 | 0.5 | 0.3 |

**この表の使い方（カメラの仕様の目安）**

| 項目 | 目安（AIの完全一致0.9以上を保つ範囲） |
|---|---|
| **板の大きさ** | 画面の幅の **30%以上**（320px幅で約100px以上） |
| **回転（傾き）** | **10度以内** |
| **透視の歪み** | **ごく小さく**（0.05以内。**正面から**撮る） |
| **ぼけ** | σ **2以内**（ピントを合わせる） |
| **動きぶれ** | **4px以内**（シャッター速度を速く・車を減速させる） |
| **明るさ・コントラスト** | 明るさ **0.3以上**、コントラスト **0.4以上**（夜間は照明が必須） |
| **隠れ** | **線1本まで**（ワイパー・枠が板に被らない位置に） |
| **ノイズ・汚れ・雨滴・JPEG圧縮** | **ほぼ問題なし**（かなり強くても大丈夫） |

> 💡 **Phase5の結論**（人手の大半は、撮影条件が悪い画像）と**つながります。** 人手を減らすには、**AIを改良する前に、上の表の条件を満たすよう、カメラ・照明・設置位置を設計する**ほうが効く。AIの**運用できる範囲**を知ることは、**システム設計の最初の仕事**です。
> ⚠ ただし、これは**合成画像での結果**（背景が単純、劣化の種類が限られる）。**実画像では、範囲はもっと狭い**と考えるべきです。

## 6. 失敗例集

**自信満々の誤り**（自動処理と判断されたのに間違い）です。val の評価データ（1,200枚）で、AIの確信度が高いのに間違えたもの（赤い枠は、検出した四隅）。

In [ ]:
show("failures_confident.png")

**見るポイント**
- ほぼ全部が、**1文字だけの取り違え**（210→216、625→626、68A→66A、か→た、な→ら、ま→さ）。**書式に合う**ので検査では防げず、**確信度も中程度（0.5〜0.77）**で、しきい値をすり抜けた。
- 画像を見ても、**人間にも読みにくい**もの（ぼけた数字、板の端が欠けたもの）が多い。
- → **Phase5の結論**: これらは、**入庫記録との照合**で、ほとんど止まる（読み間違えたナンバーは、入庫記録に存在しないため）。

## 7. しきい値の見積もりは、未使用データでも当たったか

自動処理のしきい値は、**較正セット**で「**自動処理の誤り率 1%**」を目標に決めました（昼間中心の想定）。**test**（未使用）で、実際の誤り率は？

| 難易度の混ぜ方 | 自動処理率 | 自動処理の誤り率 | 要確認率 | 読み取り不可率 | 人手対応率 | 全体の完全一致率 |
|---|---|---|---|---|---|---|
| 均等(1:1:1) | 0.5801 ±0.0563 | 0.0138 ±0.0049 | 0.3213 ±0.0481 | 0.0986 ±0.0218 | 0.4199 ±0.0563 | 0.6532 ±0.0092 |
| 昼間中心(0.6:0.3:0.1) | 0.7946 ±0.0672 | 0.0088 ±0.0022 | 0.1653 ±0.0664 | 0.0401 ±0.0039 | 0.2054 ±0.0672 | 0.8649 ±0.0119 |

- **昼間中心の想定**: 実際の誤り率は **0.88%**（目標1%）。**見積もりどおり**（標準偏差 ±0.22%）。自動処理率 **79%**、人手対応 **21%**。
- **均等（Hardが多い）**: 誤り率 **1.38%**（目標1%を超える）。**想定していた難易度の混ざり方と、実際が違う**ので、見積もりがずれる。**運用では、実際の条件の分布を、設計時に想定しておく**ことが大事。

難易度別の結果:

| 難易度 | 自動処理率 | 自動処理の誤り率 | 要確認率 | 読み取り不可率 |
|---|---|---|---|---|
| easy | 0.9240 | 0.0019 | 0.0680 | 0.0080 |
| medium | 0.7930 | 0.0231 | 0.1743 | 0.0327 |
| hard | 0.0233 | 0.1541 | 0.7217 | 0.2550 |

- **Easy**: 自動処理 92%、誤り 0.19%。ほぼ人手なしで動く。
- **Medium**: 自動処理 79%、誤り 2.3%。**残る誤りは、1文字の取り違え**。
- **Hard**: 自動処理は **2%**にとどまり、**ほぼ全部を人に回す**（正しい動作）。自動処理されたものの誤り率は高い（15%）が、**件数は少ない**。

## 8. 学び — プロジェクトの総括

### AIについて
1. **AIは、条件がそろえば強い。** 劣化したMediumを、古典の0.15から**0.90**へ。ノイズ・汚れ・逆光に強い。
2. **しかし、学習した範囲の外では、急に崩れる。** 板の大きさ・回転・透視。**学習データの条件の設計**が、AIの得意・不得意を決める（**合成データの最大の利点**は、条件を自由に設計できること）。
3. **AIは「分からない」と言えない。** 板が無くても答える。読めない画像でも答える。**確信度は粗い手がかり**にとどまった（正解と誤りは、完全には分かれない）。
4. **画像に情報が無いものは、AIにも読めない**（Hard）。**推測**ではなく**読み取り**として限界がある。

### システムについて
5. **AIは、システムの一部品。** 検査（ルール）・確信度・データベース（照合）・料金・**人の確認**が、AIの弱点を補う。
6. **書式検査は、安全で有効**（弾いたものは100%誤り）。**補正は、効果ほぼゼロ**（もっともらしい嘘を作る）。
7. **データベースとの照合が、AIの自信満々の誤りの、ほとんどを止める。** AIの外側の情報が、AIを助ける。
8. **誤課金は稀だが、起きると二次被害**がある。**近傍チェックは誤課金を0にするが、人手が2〜4倍**。**どこで折り合うかは、損害の重さ（コスト）で決まる**。運用者が決める。
9. **人手の大半は、撮影条件が悪い画像。** **AIの改良より、カメラ・照明・設置の改善が効く**。AIの**運用できる範囲**を知り、それに合わせて**現場を設計**する。

### 評価について
10. **システム全体で測らないと、本当の性能は分からない。** 検出率100%でも、読めなかった（四隅誤差）。AIの精度が良くても、システムの結果は別。
11. **しきい値は、評価とは別の、十分な数のデータで決める**（300枚では足りなかった）。
12. **最終評価は1回だけ。** 開発中の見積もりが、未使用データでも当たった（±0.01）。
13. **失敗にも、記録を残す。** データを誤って壊した事故、集計のバグ（pandasの`gt`）、説明の誤り（CTCの出力位置）。**間違いに気付き、直し、記録する**ことも、検証の一部。

### 限界と注意
- **合成画像だけ。** 書体・背景・質感は実物と違う。**sim2realギャップ**。実画像では、運用できる範囲は狭くなる。
- **ナンバーの書式は簡略化**（地名5種、かな10種、番号の表記など）。
- **人の確認は、常に正しいと仮定**。ゲートの渋滞・確認の待ち時間・夜間の無人運用は未検討。
- **プライバシー**: 実在のナンバーや、実際の車の写真は使っていない。実運用では、ナンバーは**所有者につながる情報**（取得目的・保存期間の限定が必要）。

## 🧪 試してみよう

### 演習1: 自分のカメラの仕様を決める
**やること**: 上の「運用できる範囲の目安」の表を見て、**駐車場の出入口の、カメラの設置計画**を書き出してみましょう。
- 板が画面幅の30%以上になるには、**カメラと車の距離**・**レンズの画角**・**解像度**をどうしますか？
- **夜間**の明るさ（0.3以上）と、**動きぶれ**（4px以内）を満たすには、何が必要ですか？
- **雨の日**は、どの要因が効きますか？（ヒント: 雨滴にはAIは強いが、暗さ・ぼけは？）

### 演習2: 失敗例の数字を確かめる
**やること**: 下のセルを実行し、test の最終評価の**1枚ごとの結果**（CSV）から、**「自信満々の誤り」の件数**を数えます。`decision_s0` の `"自動処理"` を、`"要確認"` に書き換えると、どうなりますか？
- **見るポイント**: 要確認に回ったもののうち、**実は正解だった割合**は？（人手が「結果的に不要だった」割合）
- **分かること**: 要確認のうち、実は正解だったのは**約15%**（805枚中124枚）。**人手の約85%は、本当に誤りを防いでいる**。一方、確信度は粗いので、**少数の正解も人に回る**（人手の無駄）。自動処理に残る誤り（1,890枚中24枚）は、見逃し。**人手の無駄**と**誤りの見逃し**のトレードオフ。

In [ ]:
f = FIN / "test_samples.csv"
if f.exists():
    t = pd.read_csv(f, keep_default_na=False)
    t["ok"] = t["p3_text_s0"] == t["gt"]
    for dec in ["自動処理", "要確認", "読み取り不可"]:             # ← "要確認" などに書き換えて試す
        s = t[t.decision_s0 == dec]
        print(f"{dec}: {len(s)}枚、そのうち正解 {int(s.ok.sum())}枚（{s.ok.mean():.1%}）、誤り {int((~s.ok).sum())}枚")

### 演習3: 再評価は、できるのか
**やること**: ターミナルで `.venv/bin/python src/final_eval.py --split test --confirm-final` を実行してみましょう（**実行しても安全**です。拒否されるだけ）。
- **見るポイント**: どんなメッセージが出ますか？ 「さらに評価したい場合」の案内は、どう書かれていますか？
- **分かること**: **testを守る仕組み**。もし、モデルを改良して再評価したいなら、**新しいシードの test セット**を作り、**手順を新たに決める**必要がある（同じtestで何度も評価すると、testも「開発用」になってしまうため）。

### 考えてみよう（コード不要）
1. この**合成画像のプロジェクト**で得た結論のうち、**実際の駐車場にも当てはまりそうなもの**と、**当てはまらなそうなもの**は？ それぞれ、なぜ？
2. もし**実画像**で試すなら、**最初に確認すべきこと**は？（ヒント: プライバシー、データの集め方、sim2realギャップ、運用できる範囲）
3. このプロジェクトで学んだ**「AIはシステムの一部品」**という見方は、**他のAI**（顔認証、翻訳、チャットAI、天気予報）に**どう応用**できますか？

---

## ✅ まとめ

- **最終評価（test）**: 完全一致は、**Easy 0.98 / Medium 0.90 / Hard 0.09**（古典は 0.71 / 0.15 / 0.00）。**開発時（val）の見積もりと±0.01以内で一致**。
- **自動処理**は、昼間中心で **79%**、**誤り率 0.88%**（目標1%、見積もりどおり）。人手対応は**21%**。
- **運用できる範囲**を、劣化要因ごとに調べた（板の大きさ30%以上、傾き10°以内、正面から、ぼけ σ2以内、など）。**学習した範囲の外では急に崩れる**。
- **AIは部品**: 検査・確信度・DB照合・人の確認が、AIの弱点を補う。**システム全体で測る**ことが大事。
- **testは1回だけ**使い、**使用済みの記録**を残した。

➡ 発展として、**学習なしで画像を読めるAI（VLM）との比較**（Phase7）を、次の [07_vlm_comparison.ipynb](07_vlm_comparison.ipynb) で行いました（合成画像のみを送る方針）。
**最終レポート**は、プロジェクト直下の [REPORT.md](../REPORT.md) にまとまっています。